# CUTEst Time

In [ ]:
import os

from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from data.CUTEst.check_CUTEst_problems import problemsToRun
from qnlab.util.method import get_methods
from qnlab.experiment.for_cutest_run import run, load_results
from qnlab.experiment.for_cutest_vis import draw_pp, individual_plot

In [ ]:
os.chdir(Path(os.path.abspath("cutest_time.ipynb")).parent.parent.resolve())
print(os.getcwd())

In [ ]:
PRECISION = 64
NOISE = np.float64(0.0)
RESULT_SUBDIR = "time"
TIME_LIMIT = 600

ERROR_CAUSING_TASKS = []

In [ ]:
problems = problemsToRun(PRECISION)
methods, ALGORITHM_COLORS, ALGORITHM_LINE_STYLES = get_methods()
methods = [m for m in methods if m[0].base in ["NTRQN", "SciPy"]]

run(
    problems,
    methods,
    PRECISION,
    NOISE,
    ERROR_CAUSING_TASKS,
    TL=TIME_LIMIT,
    result_subdir=RESULT_SUBDIR,
)

In [ ]:
for _gtol in [1e-1, 1e-3, 1e-5]:
    gtol = np.float64(_gtol)
    problems = problemsToRun(PRECISION)

    alg_names, timesM, fxsM, gnormsM, problems = load_results(
        methods,
        problems,
        PRECISION,
        NOISE,
        gtol,
        metric="time",
        result_subdir=RESULT_SUBDIR,
    )
    draw_pp(
        alg_names,
        timesM,
        ALGORITHM_COLORS,
        ALGORITHM_LINE_STYLES,
        PRECISION,
        NOISE,
        gtol,
        metric="time",
    )

    if gtol == 1e-5:
        individual_plot(
            problems,
            methods,
            PRECISION,
            NOISE,
            x_axis="time",
            result_subdir=RESULT_SUBDIR,
        )

    data = {"problem": problems}
    for i, alg_name in enumerate(alg_names):
        data[alg_name] = timesM[i, :].tolist()
    df = pd.DataFrame(data)
    df.set_index("problem", inplace=True)

    def color_scale_with_cmap(row):
        if np.all(np.isinf(row.values)):
            return ["background-color: rgba(0, 0, 0, 0.8)" for _ in row.values]
        norm = plt.Normalize(vmin=row.min(), vmax=row.min() * 10)  # type:ignore
        cmap = matplotlib.colormaps["coolwarm"]
        return [
            f"background-color: rgba({int(r * 255)}, {int(g * 255)}, {int(b * 255)}, 0.8)"
            for r, g, b, _ in cmap(norm(row.values))
        ]

    styled_df = df.style.format("{:.3e}").apply(color_scale_with_cmap, axis=1)
    display(styled_df)